# BIST Walk-Forward Strateji Yarışması
## Klasik TA · GARCH · XGBoost · HMM (Simons) · Ensemble

### Neden Walk-Forward?

| Yöntem | Sorun |
|---|---|
| Basit train/test split | Modeller geçmişi "görür" → gerçekçi değil |
| **Walk-Forward (WFO)** | Her fold sadece **o ana kadar olan veriyi** görür ✓ |

```
Fold 1:  [████ TRAIN ████]  [TEST ]
Fold 2:  [██████ TRAIN ██████]  [TEST ]
Fold 3:  [████████ TRAIN ████████]  [TEST ]
...
         ← expanding pencere →
```

**Kural**: Test foldundaki tahmin, o tarihte gerçek pozisyon alınmış gibi değerlendirilir.
Model bir sonraki foldu görmeden eğitilir → **gerçek dünya simülasyonu**.

---
**Adımlar**
1. Özellik mühendisliği (RSI, MACD, Hacim, Göreceli Güç)
2. 5 strateji sınıfı: TA · GARCH · XGBoost · HMM · Ensemble
3. Walk-Forward motor (expanding pencere, 63 günlük test foldu)
4. İstatistiksel karşılaştırma (Bootstrap Sharpe CI, t-test)
5. Şampiyon → tüm veriye fit → **bugünkü sinyal**

In [ ]:
import subprocess, sys

def _install(pkg, label=None):
    label = label or pkg
    print(f"  {label}...", end=" ", flush=True)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       capture_output=True, text=True)
    print("OK" if r.returncode == 0 else f"HATA → {r.stderr[-120:]}")

for pkg in [
    "pandas", "numpy", "scipy", "scikit-learn",
    "xgboost",     # XGBoost sınıflandırıcı
    "arch",        # GARCH(1,1) — arch-py
    "hmmlearn",    # Gizli Markov Modeli (Simons)
    "matplotlib",  # Görselleştirme
    "yfinance",    # Gerçek veri (opsiyonel)
]:
    _install(pkg)

print("\nKütüphaneler hazır.")

In [ ]:
import warnings, abc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:.3f}")
plt.rcParams.update({"figure.figsize": (15, 5), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10})

# ── Walk-Forward Parametreleri ───────────────────────────────────────────────
WF_TRAIN_MIN  = 252   # Başlangıç eğitim penceresi (1 yıl minimum)
WF_TEST_DAYS  = 63    # Her fold test uzunluğu (~3 ay)
WF_STEP       = 63    # Expanding: her adımda 63 gün ilerle

# ── Teknik Analiz Parametreleri ──────────────────────────────────────────────
RSI_PERIOD    = 14
MACD_FAST, MACD_SLOW, MACD_SIGNAL = 12, 26, 9
VOL_WINDOW    = 20

# ── GARCH Parametreleri ──────────────────────────────────────────────────────
GARCH_LOW_Q   = 0.35   # Düşük vol eşiği (alt %35 percentile)
GARCH_HIGH_Q  = 0.70   # Yüksek vol eşiği (üst %30 percentile)
MOM_WINDOW    = 5

# ── XGBoost Özellikleri ──────────────────────────────────────────────────────
FEATURES = ["RSI", "MACD", "MACD_signal", "MACD_hist", "Vol_ratio", "Rel_strength",
            "ret_1d", "ret_5d", "ret_20d"]  # Ek lag getiri özellikleri

STRATEGY_COLORS = {
    "A_KlasikTA"  : "#2196F3",
    "B_GARCH"     : "#FF9800",
    "C_XGBoost"   : "#4CAF50",
    "D_HMM"       : "#9C27B0",
    "E_Ensemble"  : "#F44336",
    "BuyHold"     : "#9E9E9E",
}

print("Konfigürasyon yüklendi.")
print(f"  WF_TRAIN_MIN={WF_TRAIN_MIN} | WF_TEST_DAYS={WF_TEST_DAYS} | WF_STEP={WF_STEP}")
print(f"  Özellikler: {FEATURES}")

## Bölüm 1 — Özellik Mühendisliği

In [ ]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Tüm teknik göstergeler ve etiket hesaplanır.

    Not: Özellikler SADECE geçmiş veriye bakar (lookback pencereler).
    Label = Bir sonraki günün log getirisi > 0 ise 1, değilse 0.
    """
    df = df.copy().sort_index()

    # ── Log Getiri ───────────────────────────────────────────────────────────
    df["log_ret"] = np.log(df["Close"] / df["Close"].shift(1))

    # ── Hedef Değişken ───────────────────────────────────────────────────────
    df["Label"] = (df["log_ret"].shift(-1) > 0).astype(float)

    # ── RSI(14) ──────────────────────────────────────────────────────────────
    d     = df["Close"].diff()
    gain  = d.where(d > 0, 0.0).rolling(RSI_PERIOD).mean()
    loss  = (-d).where(d < 0, 0.0).rolling(RSI_PERIOD).mean()
    df["RSI"] = 100 - (100 / (1 + gain / (loss + 1e-9)))

    # ── MACD(12,26,9) ────────────────────────────────────────────────────────
    ema_f            = df["Close"].ewm(span=MACD_FAST, adjust=False).mean()
    ema_s            = df["Close"].ewm(span=MACD_SLOW, adjust=False).mean()
    df["MACD"]       = ema_f - ema_s
    df["MACD_signal"]= df["MACD"].ewm(span=MACD_SIGNAL, adjust=False).mean()
    df["MACD_hist"]  = df["MACD"] - df["MACD_signal"]

    # ── Hacim Oranı ──────────────────────────────────────────────────────────
    df["Vol_ratio"]   = df["Volume"] / (df["Volume"].rolling(VOL_WINDOW).mean() + 1e-9)

    # ── Endeks Göreceli Gücü ─────────────────────────────────────────────────
    df["Rel_strength"]= df["Close"] / (df["Endeks_Close"] + 1e-9)

    # ── Lag Getiriler ────────────────────────────────────────────────────────
    df["ret_1d"]  = df["log_ret"]
    df["ret_5d"]  = np.log(df["Close"] / df["Close"].shift(5))
    df["ret_20d"] = np.log(df["Close"] / df["Close"].shift(20))

    df.dropna(inplace=True)
    print(f"  Özellik mühendisliği: {len(df)} satır, {df.shape[1]} sütun")
    return df

## Bölüm 2 — Strateji Sınıfları (fit / predict arayüzü)

In [ ]:
# ── Temel Arayüz ─────────────────────────────────────────────────────────────
class BaseStrategy(abc.ABC):
    """fit(df_train) → predict(df_test) → pd.Series[int] arayüzü."""

    @abc.abstractmethod
    def fit(self, df_train: pd.DataFrame) -> None: ...

    @abc.abstractmethod
    def predict(self, df_test: pd.DataFrame) -> pd.Series: ...

    @property
    def name(self) -> str:
        return self.__class__.__name__


# ═══════════════════════════════════════════════════════════════════════════════
# STRATEJİ A — Klasik Teknik Analiz (RSI + MACD)
# ═══════════════════════════════════════════════════════════════════════════════
class ClassicTAStrategy(BaseStrategy):
    """
    Stateless kural tabanlı strateji.
    Pozisyon = 1 (Long) eğer:
      • MACD histogramı pozitif (MACD > sinyal) VE
      • RSI aşırı alım bölgesinde değil (30 < RSI < 72)
    """
    name = "A_KlasikTA"

    def fit(self, df_train: pd.DataFrame) -> None:
        pass   # Kural tabanlı, parametresiz

    def predict(self, df_test: pd.DataFrame) -> pd.Series:
        macd_bull = df_test["MACD_hist"] > 0
        rsi_ok    = (df_test["RSI"] > 30) & (df_test["RSI"] < 72)
        pos       = (macd_bull & rsi_ok).astype(int)
        return pos.rename("pos")


# ═══════════════════════════════════════════════════════════════════════════════
# STRATEJİ B — GARCH(1,1) Oynaklık Rejimi
# ═══════════════════════════════════════════════════════════════════════════════
class GARCHStrategy(BaseStrategy):
    """
    fit  : GARCH(1,1) eğitir; volatilite eşiklerini hesaplar.
    predict : GARCH rekürsiyon formülüyle test setinin koşullu
              volatilitesini tahmin eder; düşük-vol + pozitif
              momentum → Long, yüksek-vol → Flat.
    """
    name = "B_GARCH"

    def __init__(self):
        self._omega = 0.05
        self._alpha = 0.05
        self._beta  = 0.90
        self._low   = None
        self._high  = None
        self._h_last   = 1.0
        self._eps2_last = 0.01

    def fit(self, df_train: pd.DataFrame) -> None:
        from arch import arch_model
        ret_pct = (df_train["log_ret"] * 100).dropna()

        try:
            garch  = arch_model(ret_pct, vol="Garch", p=1, q=1,
                                mean="Zero", dist="normal")
            result = garch.fit(
                starting_values=np.array([self._omega, self._alpha, self._beta]),
                disp="off", show_warning=False, options={"maxiter": 300}
            )
            p = result.params
            self._omega = max(float(p.get("omega",     0.05)), 1e-9)
            self._alpha = max(float(p.get("alpha[1]",  0.05)), 1e-9)
            self._beta  = max(float(p.get("beta[1]",   0.90)), 1e-9)

            cv             = result.conditional_volatility
            self._low      = float(cv.quantile(GARCH_LOW_Q))
            self._high     = float(cv.quantile(GARCH_HIGH_Q))
            self._h_last   = float(cv.iloc[-1]) ** 2
            self._eps2_last= float(ret_pct.iloc[-1]) ** 2

        except Exception:
            # Yedek: EWMA oynaklık
            ewm = ret_pct.ewm(span=20).std()
            self._low  = float(ewm.quantile(GARCH_LOW_Q))
            self._high = float(ewm.quantile(GARCH_HIGH_Q))
            self._h_last = float(ewm.iloc[-1]) ** 2
            self._eps2_last = float(ret_pct.iloc[-1]) ** 2

    def predict(self, df_test: pd.DataFrame) -> pd.Series:
        if self._low is None:
            return pd.Series(0, index=df_test.index)

        ret_pct = (df_test["log_ret"] * 100).values
        mom     = df_test["Close"].pct_change(MOM_WINDOW).values

        # GARCH(1,1) rekürsiyon: h_t = ω + α·ε²_{t-1} + β·h_{t-1}
        h, prev_eps2 = self._h_last, self._eps2_last
        vols = []
        for eps in ret_pct:
            h = self._omega + self._alpha * prev_eps2 + self._beta * h
            h = max(h, 1e-9)
            vols.append(np.sqrt(h))
            prev_eps2 = eps ** 2
        self._h_last    = h
        self._eps2_last = prev_eps2

        pos = []
        for v, m in zip(vols, mom):
            m = 0.0 if np.isnan(m) else m
            if v > self._high:
                pos.append(0)
            elif v < self._low and m > 0:
                pos.append(1)
            else:
                pos.append(1 if m > 0 else 0)

        return pd.Series(pos, index=df_test.index, dtype=int, name="pos")


# ═══════════════════════════════════════════════════════════════════════════════
# STRATEJİ C — XGBoost İkili Sınıflandırıcı
# ═══════════════════════════════════════════════════════════════════════════════
class XGBoostStrategy(BaseStrategy):
    """
    fit  : Train verisiyle XGBClassifier eğitir.
    predict : Test özelliklerini sınıflandırır (0/1).
    Accuracy fold bazında raporlanır.
    """
    name = "C_XGBoost"

    def __init__(self):
        self.model     = None
        self.last_acc  = None
        self._fold_accs = []

    def fit(self, df_train: pd.DataFrame) -> None:
        valid = df_train.dropna(subset=FEATURES + ["Label"])
        X = valid[FEATURES].values
        y = valid["Label"].astype(int).values

        self.model = XGBClassifier(
            n_estimators=150, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="logloss", random_state=42, verbosity=0
        )
        self.model.fit(X, y)

    def predict(self, df_test: pd.DataFrame) -> pd.Series:
        if self.model is None:
            return pd.Series(0, index=df_test.index)
        valid = df_test.dropna(subset=FEATURES)
        if valid.empty:
            return pd.Series(0, index=df_test.index)
        preds = self.model.predict(valid[FEATURES].values)
        pos   = pd.Series(0, index=df_test.index, dtype=int, name="pos")
        pos.loc[valid.index] = preds

        # Fold doğruluğu (label biliniyorsa)
        if "Label" in df_test.columns:
            lbls = df_test["Label"].loc[valid.index].astype(int)
            self.last_acc = accuracy_score(lbls, preds)
            self._fold_accs.append(self.last_acc)
        return pos

    @property
    def avg_accuracy(self):
        return np.mean(self._fold_accs) if self._fold_accs else None


# ═══════════════════════════════════════════════════════════════════════════════
# STRATEJİ D — HMM Rejim Tespiti (Jim Simons İlhamı)
# ═══════════════════════════════════════════════════════════════════════════════
class HMMStrategy(BaseStrategy):
    """
    GaussianHMM ile Bull/Bear rejim tespiti.
    Özellikler: günlük getiri, kısa vadeli volatilite, hacim değişimi.
    Bull durumunda Long, diğer durumlarda Flat.
    """
    name = "D_HMM"

    def __init__(self, n_states: int = 3):
        self.n_states   = n_states
        self.model      = None
        self.bull_state = None
        self._scaler_mean = None
        self._scaler_std  = None

    def _build_features(self, df: pd.DataFrame) -> np.ndarray:
        ret    = df["log_ret"].values
        vol5   = pd.Series(ret).rolling(5).std().fillna(method="bfill").values
        volr   = df["Vol_ratio"].fillna(1.0).values
        return np.column_stack([ret, vol5, volr])

    def fit(self, df_train: pd.DataFrame) -> None:
        try:
            from hmmlearn.hmm import GaussianHMM
        except ImportError:
            self.model = None
            return

        X = self._build_features(df_train)
        self._scaler_mean = X.mean(axis=0)
        self._scaler_std  = X.std(axis=0) + 1e-9
        X_sc = (X - self._scaler_mean) / self._scaler_std

        self.model = GaussianHMM(
            n_components=self.n_states, covariance_type="full",
            n_iter=200, random_state=42
        )
        self.model.fit(X_sc)

        # Bull durumu = en yüksek ortalama getirili durum
        states  = self.model.predict(X_sc)
        ret_arr = df_train["log_ret"].values
        state_ret = {
            s: ret_arr[states == s].mean() if (states == s).sum() > 0 else -999
            for s in range(self.n_states)
        }
        self.bull_state = max(state_ret, key=state_ret.get)

    def predict(self, df_test: pd.DataFrame) -> pd.Series:
        if self.model is None or self._scaler_mean is None:
            return pd.Series(0, index=df_test.index, dtype=int, name="pos")
        try:
            X = self._build_features(df_test)
            X_sc = (X - self._scaler_mean) / self._scaler_std
            states = self.model.predict(X_sc)
            pos    = (states == self.bull_state).astype(int)
        except Exception:
            pos = np.zeros(len(df_test), dtype=int)
        return pd.Series(pos, index=df_test.index, name="pos")


# ═══════════════════════════════════════════════════════════════════════════════
# STRATEJİ E — Çoğunluk Oyu Ensemble
# ═══════════════════════════════════════════════════════════════════════════════
class EnsembleStrategy(BaseStrategy):
    """
    A, B, C, D stratejilerini çalıştırır; çoğunluk oyuyla (≥ 2/4) karar verir.
    """
    name = "E_Ensemble"

    def __init__(self):
        self._base = [
            ClassicTAStrategy(),
            GARCHStrategy(),
            XGBoostStrategy(),
            HMMStrategy(),
        ]

    def fit(self, df_train: pd.DataFrame) -> None:
        for s in self._base:
            try:
                s.fit(df_train)
            except Exception:
                pass

    def predict(self, df_test: pd.DataFrame) -> pd.Series:
        votes = pd.DataFrame(index=df_test.index)
        for s in self._base:
            try:
                votes[s.name] = s.predict(df_test).values
            except Exception:
                votes[s.name] = 0
        # Çoğunluk: 4 strateji → ≥ 2 oy gerekli
        pos = (votes.sum(axis=1) >= 2).astype(int)
        return pos.rename("pos")


STRATEGIES = {
    "A_KlasikTA"  : ClassicTAStrategy(),
    "B_GARCH"     : GARCHStrategy(),
    "C_XGBoost"   : XGBoostStrategy(),
    "D_HMM"       : HMMStrategy(),
    "E_Ensemble"  : EnsembleStrategy(),
}

print("5 strateji sınıfı tanımlandı:")
for nm in STRATEGIES:
    print(f"  • {nm}")

## Bölüm 3 — Walk-Forward Optimizasyon Motoru

In [ ]:
def walk_forward_engine(
    df_feat       : pd.DataFrame,
    strategies    : dict,
    train_min     : int = WF_TRAIN_MIN,
    test_days     : int = WF_TEST_DAYS,
    step          : int = WF_STEP,
    verbose       : bool = True,
) -> dict:
    """
    Expanding-window walk-forward backtest.

    Her foldda:
      1. [0 : train_end]  → strategy.fit()
      2. [train_end : train_end+test_days]  → strategy.predict()
      3. train_end += step

    Veri sızıntısı yoktur: test foldundaki veriler hiçbir zaman
    eğitim sırasında kullanılmaz.

    Çıktı: {strategy_name: pd.Series[int]}  (tüm test periodları birleşik)
    """
    n      = len(df_feat)
    # Her strateji için sıfır pozisyon
    positions = {nm: np.zeros(n, dtype=int) for nm in strategies}

    fold      = 0
    train_end = train_min

    while train_end + test_days <= n:
        test_end   = min(train_end + test_days, n)
        train_df   = df_feat.iloc[:train_end]          # Expanding: 0 → train_end
        test_df    = df_feat.iloc[train_end:test_end]  # Bilinmeyen dönem

        if verbose and fold % 3 == 0:
            t0 = str(df_feat.index[train_end])[:10]
            t1 = str(df_feat.index[test_end-1])[:10]
            print(f"  Fold {fold+1:02d}: eğitim={train_end}g | test={t0}→{t1} "
                  f"({test_end-train_end}g)", flush=True)

        for nm, strategy in strategies.items():
            try:
                strategy.fit(train_df)
                pos = strategy.predict(test_df)
                positions[nm][train_end:test_end] = pos.values[:test_end-train_end]
            except Exception as exc:
                if verbose:
                    print(f"    [{nm}] fold {fold+1} hatası: {exc}")

        fold      += 1
        train_end += step

    if verbose:
        print(f"\nToplam fold: {fold} | "
              f"WFO kapsanan dönem: "
              f"{str(df_feat.index[train_min])[:10]} → "
              f"{str(df_feat.index[min(train_min+fold*step-1, n-1)])[:10]}")

    return {nm: pd.Series(arr, index=df_feat.index)
            for nm, arr in positions.items()}


print("Walk-Forward motoru hazır.")

## Bölüm 4 — Metrik Hesaplama & İstatistiksel Testler

In [ ]:
def calc_metrics(
    position   : pd.Series,
    log_returns: pd.Series,
    wf_start   : int,
    name       : str = "",
    n_boot     : int = 800,
) -> dict:
    """
    WFO test dönemine ait performans metrikleri + bootstrap Sharpe CI.

    Gecikme  : Sinyal gün-sonu alınır, ertesi gün açılışta uygulanır (1G lag).
    Pozisyon : 1=Long, 0=Flat (açığa satış yok, komisyon yok)
    """
    pos  = position.iloc[wf_start:].shift(1).fillna(0)  # 1 gün gecikme
    ret  = log_returns.iloc[wf_start:]
    strat = pos * ret  # Strateji günlük log getirileri

    # ── Temel Metrikler ──────────────────────────────────────────────────────
    total_ret   = float(np.expm1(strat.sum()) * 100)
    n_days_wf   = len(strat)
    ann_factor  = 252
    daily_mean  = strat.mean()
    daily_std   = strat.std()
    sharpe      = float(daily_mean / daily_std * np.sqrt(ann_factor)) if daily_std > 1e-9 else 0.0

    # ── Maksimum Drawdown ─────────────────────────────────────────────────────
    cum        = np.exp(strat.cumsum())
    peak       = cum.cummax()
    dd         = (cum - peak) / peak
    max_dd     = float(dd.min() * 100)
    calmar     = float(total_ret / abs(max_dd)) if abs(max_dd) > 1e-3 else 0.0

    # ── Win Rate ─────────────────────────────────────────────────────────────
    active     = strat[pos > 0]
    win_rate   = float((active > 0).mean() * 100) if len(active) > 0 else 0.0

    # ── Bootstrap Sharpe CI (%95) ─────────────────────────────────────────────
    rng        = np.random.default_rng(42)
    boot_sh    = []
    arr        = strat.values
    for _ in range(n_boot):
        sample = rng.choice(arr, size=len(arr), replace=True)
        m, s   = sample.mean(), sample.std()
        boot_sh.append((m / s * np.sqrt(ann_factor)) if s > 1e-9 else 0.0)
    ci_low  = float(np.percentile(boot_sh, 2.5))
    ci_high = float(np.percentile(boot_sh, 97.5))

    # ── t-test: Strateji getirileri sıfırdan farklı mı? ──────────────────────
    if len(active) > 10:
        t_stat, p_val = stats.ttest_1samp(active.values, 0)
    else:
        t_stat, p_val = 0.0, 1.0

    # ── Pozisyon istatistikleri ───────────────────────────────────────────────
    n_trades = int((pos.diff().fillna(0) > 0).sum())
    n_pos    = int((pos > 0).sum())
    exposure = float(n_pos / n_days_wf * 100) if n_days_wf > 0 else 0.0

    return {
        "Getiri (%)"       : round(total_ret, 2),
        "Sharpe"           : round(sharpe, 3),
        "Sharpe CI"        : f"[{ci_low:.2f}, {ci_high:.2f}]",
        "MaxDD (%)"        : round(max_dd, 2),
        "Calmar"           : round(calmar, 3),
        "WinRate (%)"      : round(win_rate, 2),
        "Exposure (%)"     : round(exposure, 1),
        "t-stat"           : round(float(t_stat), 3),
        "p-value"          : round(float(p_val), 4),
        "İşlem Sayısı"     : n_trades,
        "_strat_ret"       : strat,   # Grafik için
    }


def champion_score(row: pd.Series) -> float:
    """
    Çok kriterli şampiyon skoru.
    Getiri %45 + WinRate %30 + Sharpe %25
    p-value < 0.10 olmayan stratejiyi cezalandır.
    """
    score = row["Getiri (%)"] * 0.45 + row["WinRate (%)"] * 0.30 + row["Sharpe"] * 0.25
    if row["p-value"] > 0.10:
        score *= 0.8   # İstatistiksel anlamsız → %20 ceza
    return score


print("Metrik fonksiyonları hazır.")

In [ ]:
def plot_wfo_results(
    strat_rets: dict,
    df_comparison: pd.DataFrame,
    champion: str,
    wf_start_date,
):
    fig, axes = plt.subplots(1, 3, figsize=(19, 5))

    # ── Sol: Kümülatif Getiri ─────────────────────────────────────────────────
    ax1 = axes[0]
    ls  = {nm: ("-" if nm == champion else "--") for nm in strat_rets}
    lw  = {nm: (2.5 if nm == champion else 1.4) for nm in strat_rets}

    for nm, sr in strat_rets.items():
        cum = np.exp(sr.cumsum())
        pct = (cum.iloc[-1] - 1) * 100
        lbl = f"{'★ ' if nm==champion else ''}{nm} ({pct:+.1f}%)"
        ax1.plot(cum.index, cum.values,
                 label=lbl, color=STRATEGY_COLORS.get(nm, "black"),
                 ls=ls[nm], lw=lw[nm])

    ax1.axhline(1, color="black", lw=0.7, ls=":", alpha=0.5)
    ax1.set_title("Kümülatif Getiri (WFO Test Dönemi)", fontweight="bold")
    ax1.set_ylabel("Portföy (Başlangıç = 1)")
    ax1.legend(fontsize=8, loc="upper left")
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.2f}"))

    # ── Orta: Sharpe Karşılaştırması (CI ile) ────────────────────────────────
    ax2 = axes[1]
    strat_names = [n for n in df_comparison.index if n != "BuyHold"]
    sharpes     = df_comparison.loc[strat_names, "Sharpe"]
    ci_low      = [float(df_comparison.loc[n,"Sharpe CI"].split(",")[0].strip("[")) for n in strat_names]
    ci_high     = [float(df_comparison.loc[n,"Sharpe CI"].split(",")[1].strip("]").strip()) for n in strat_names]
    errors      = [
        [sharpes[n] - ci_low[i], ci_high[i] - sharpes[n]]
        for i, n in enumerate(strat_names)
    ]
    x    = np.arange(len(strat_names))
    cols = [STRATEGY_COLORS.get(n, "gray") for n in strat_names]
    errs = np.array([[e[0] for e in errors], [e[1] for e in errors]])
    ax2.bar(x, sharpes, color=cols, alpha=0.8, edgecolor="white")
    ax2.errorbar(x, sharpes, yerr=errs, fmt="none", color="black", capsize=4, lw=1.5)
    ax2.axhline(0, color="black", lw=0.8)
    ax2.set_xticks(x)
    ax2.set_xticklabels(strat_names, rotation=20, ha="right", fontsize=8)
    ax2.set_title("Sharpe Oranı (Bootstrap %95 CI)", fontweight="bold")
    ax2.set_ylabel("Sharpe")

    # ── Sağ: Getiri vs WinRate scatter ───────────────────────────────────────
    ax3 = axes[2]
    for nm in strat_names:
        row  = df_comparison.loc[nm]
        col  = STRATEGY_COLORS.get(nm, "gray")
        star = "★ " if nm == champion else ""
        ax3.scatter(row["WinRate (%)"], row["Getiri (%)"],
                    s=120, color=col, zorder=5, edgecolors="white", lw=1.5)
        ax3.annotate(f"{star}{nm}", (row["WinRate (%)"], row["Getiri (%)"]),
                     textcoords="offset points", xytext=(6, 3), fontsize=7.5)

    ax3.axhline(0, color="black", lw=0.8, ls="--", alpha=0.5)
    ax3.set_xlabel("Win Rate (%)")
    ax3.set_ylabel("Toplam Getiri (%)")
    ax3.set_title("Getiri vs Win Rate (WFO)", fontweight="bold")

    plt.suptitle(
        f"Walk-Forward Test Başlangıcı: {str(wf_start_date)[:10]}  |  "
        f"Şampiyon: {champion}",
        fontsize=11, y=1.01
    )
    plt.tight_layout()
    plt.savefig("/tmp/wfo_karsilastirma.png", dpi=130, bbox_inches="tight")
    plt.show()
    print("Grafik: /tmp/wfo_karsilastirma.png")

## Bölüm 5 — Walk-Forward Çalıştırma & Kıyaslama

In [ ]:
def run_wfo_comparison(df: pd.DataFrame) -> tuple:
    """
    Tam walk-forward analiz pipeline'ı.

    1. Özellik mühendisliği
    2. WFO motor → 5 strateji
    3. Metrik hesaplama + Bootstrap CI
    4. Şampiyon belirleme (istatistiksel skor)
    5. Sonuç tablosu + görselleştirme

    Döndürür
    --------
    (df_results, champion_name, wfo_positions, df_feat)
    """
    SEP = "=" * 72

    print(SEP)
    print("  BIST WALK-FORWARD STRATEJİ YARIŞMASI")
    print("  Her fold sadece GEÇMİŞ veriyle eğitilir → Gerçekçi test")
    print(SEP)

    # ── 1. Özellik mühendisliği ──────────────────────────────────────────────
    print("\n[1/3] Özellik mühendisliği...")
    df_f = feature_engineering(df)
    n    = len(df_f)
    n_folds_est = max(1, (n - WF_TRAIN_MIN) // WF_STEP)
    print(f"  Toplam: {n} gün | Tahmini fold sayısı: ~{n_folds_est}")
    print(f"  WFO test başlangıcı: {str(df_f.index[WF_TRAIN_MIN])[:10]}")

    # ── 2. Walk-Forward ──────────────────────────────────────────────────────
    print(f"\n[2/3] Walk-Forward başlatılıyor...")
    print(f"  GARCH fold başına ~5-10 sn; toplam ~{n_folds_est*7//60+1} dk beklenir\n")

    wfo_pos = walk_forward_engine(
        df_feat  = df_f,
        strategies = STRATEGIES,
        train_min  = WF_TRAIN_MIN,
        test_days  = WF_TEST_DAYS,
        step       = WF_STEP,
        verbose    = True,
    )

    # Buy & Hold benchmark
    bh_sr = df_f["log_ret"].copy()
    bh_sr.iloc[:WF_TRAIN_MIN] = 0   # WFO başlamadan önce sıfır

    # ── 3. Metrikler ─────────────────────────────────────────────────────────
    print("\n[3/3] Metrikler hesaplanıyor (Bootstrap CI)...")
    results    = {}
    strat_rets = {}

    for nm, pos in wfo_pos.items():
        m = calc_metrics(pos, df_f["log_ret"], WF_TRAIN_MIN, nm)
        strat_rets[nm] = m.pop("_strat_ret")
        results[nm]    = m
        sig = "✓" if m["p-value"] < 0.10 else "✗"
        print(f"  {nm:<14} Getiri:{m['Getiri (%)']:+7.2f}%  "
              f"Sharpe:{m['Sharpe']:6.3f}  WinRate:{m['WinRate (%)']:5.1f}%  "
              f"p={m['p-value']:.3f}{sig}")

    # Benchmark
    bh_m = calc_metrics(pd.Series(1, index=df_f.index), df_f["log_ret"], WF_TRAIN_MIN, "BuyHold")
    strat_rets["BuyHold"] = bh_m.pop("_strat_ret")
    results["BuyHold"]    = bh_m

    # ── Tablo ────────────────────────────────────────────────────────────────
    cols_show = ["Getiri (%)", "Sharpe", "Sharpe CI", "MaxDD (%)",
                 "Calmar", "WinRate (%)", "Exposure (%)", "p-value", "İşlem Sayısı"]
    df_res = pd.DataFrame(results).T[cols_show]
    df_res.index.name = "Strateji"

    print("\n" + SEP)
    print(f"  PERFORMANS TABLOSU — WFO Test Dönemi "
          f"({n-WF_TRAIN_MIN} gün | ~{n_folds_est} fold)")
    print(f"  * Tüm metrikler gerçek out-of-sample; veri sızıntısı yok")
    print(SEP)
    print(df_res.to_string())

    # ── Şampiyon ─────────────────────────────────────────────────────────────
    strat_df = df_res.drop(index=["BuyHold"], errors="ignore").copy()
    strat_df["_score"] = strat_df.apply(champion_score, axis=1)
    champion  = strat_df["_score"].idxmax()
    champ_row = df_res.loc[champion]

    print("\n" + SEP)
    print(f"\n  ★★★ ŞAMPİYON STRATEJİ: {champion} ★★★")
    print()
    print(f"  Toplam Getiri  : {champ_row['Getiri (%)']:+.2f}%  "
          f"(Benchmark: {results['BuyHold']['Getiri (%)']:+.2f}%)")
    print(f"  Sharpe Oranı   :  {champ_row['Sharpe']:.3f}  "
          f"CI: {champ_row['Sharpe CI']}")
    print(f"  Max Drawdown   : {champ_row['MaxDD (%)']:.2f}%")
    print(f"  Calmar Oranı   :  {champ_row['Calmar']:.3f}")
    print(f"  Win Rate       : {champ_row['WinRate (%)']:.1f}%")
    print(f"  p-value        :  {champ_row['p-value']:.4f}  "
          f"({'İstatistiksel olarak anlamlı ✓' if champ_row['p-value']<0.10 else 'Dikkat: anlamsız ✗'})")
    print("\n" + SEP)

    # Görsel
    plot_wfo_results(strat_rets, df_res, champion, df_f.index[WF_TRAIN_MIN])

    return df_res, champion, wfo_pos, df_f

## Bölüm 6 — Şampiyon Strateji ile Güncel Sinyal

In [ ]:
def generate_current_signal(
    df_feat: pd.DataFrame,
    champion_name: str,
    strategies: dict,
) -> dict:
    """
    Şampiyon stratejiyi TÜM geçmiş veriye fit eder ve
    en son gün için bir sonraki işlem günü sinyalini üretir.

    Not: Label sütunundaki son satır NaN olabilir (Label = bir sonraki
    günün yönü, henüz bilinmiyor). Eğitim için bu satır çıkarılır;
    ancak özellik değerleri geçerlidir.
    """
    SEP2 = "─" * 60

    # Şampiyon stratejinin temiz kopyasını al
    champ = STRATEGIES[champion_name]

    # Son satırı test için ayır (sadece özellikler bilinir, label bilinmez)
    valid_train = df_feat.dropna(subset=["Label"])
    last_row    = df_feat.iloc[[-1]]   # Güncel son bar

    print(SEP2)
    print(f"  Şampiyon ({champion_name}) tüm veriye fit ediliyor...")
    champ.fit(valid_train)
    print(f"  Fit tamamlandı. ({len(valid_train)} gün)")

    pos_pred = champ.predict(last_row)
    signal   = int(pos_pred.iloc[0])
    last_date= str(df_feat.index[-1])[:10]

    # XGBoost ise olasılık ver
    prob = None
    if hasattr(champ, "model") and champ.model is not None and hasattr(champ.model, "predict_proba"):
        try:
            feat_vals = last_row[FEATURES].values
            proba = champ.model.predict_proba(feat_vals)[0]
            prob  = float(proba[1])   # P(Yükseliş)
        except Exception:
            prob = None

    # HMM ise rejim olasılığı
    hmm_regime = None
    if champion_name == "D_HMM" and hasattr(champ, "model") and champ.model is not None:
        try:
            from hmmlearn.hmm import GaussianHMM
            X = champ._build_features(last_row)
            X_sc = (X - champ._scaler_mean) / champ._scaler_std
            proba_hmm  = champ.model.predict_proba(X_sc)[0]
            bull_p = float(proba_hmm[champ.bull_state])
            hmm_regime = bull_p
        except Exception:
            pass

    print()
    print(SEP2)
    print(f"  SON VERİ TARİHİ  : {last_date}")
    print(f"  STRATEJİ         : {champion_name}")
    print()
    if signal == 1:
        print("  ██████████████████████████████████████")
        print("  ██  SONRAKI GÜN SİNYALİ: AL  (LONG)  ██")
        print("  ██████████████████████████████████████")
    else:
        print("  ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░")
        print("  ░░  SONRAKI GÜN SİNYALİ: FLAT (NAKİT)  ░░")
        print("  ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░")
    print()
    if prob is not None:
        print(f"  Yükseliş olasılığı : {prob*100:.1f}%")
        print(f"  Düşüş  olasılığı   : {(1-prob)*100:.1f}%")
    if hmm_regime is not None:
        print(f"  Bull rejim olasılığı: {hmm_regime*100:.1f}%")
    print(SEP2)

    # Güncel teknik göstergeler
    latest = df_feat.iloc[-1]
    print("\n  Güncel Teknik Göstergeler:")
    print(f"    RSI(14)       : {latest.get('RSI', float('nan')):.1f}")
    print(f"    MACD hist     : {latest.get('MACD_hist', float('nan')):.4f}")
    print(f"    Hacim oranı   : {latest.get('Vol_ratio', float('nan')):.2f}x")
    print(f"    Göreceli güç  : {latest.get('Rel_strength', float('nan')):.4f}")
    print(f"    5G getiri     : {latest.get('ret_5d', float('nan'))*100:.2f}%")
    print(SEP2)

    return {"signal": signal, "prob_up": prob, "date": last_date,
            "champion": champion_name}


# Örnek kullanım (run_wfo_comparison çalıştırıldıktan sonra):
# signal_info = generate_current_signal(df_feat, champion, STRATEGIES)

## Demo — Sentetik Veri ile Uçtan Uca Test

In [ ]:
def create_synthetic_bist(n_days=1500, seed=42) -> pd.DataFrame:
    """
    Geometric Brownian Motion + volatilite kümelenmesi + hacim korelasyonu.
    Gerçek BIST hisselerinin istatistiksel özelliklerine yakın sentetik veri.
    """
    rng    = np.random.default_rng(seed)
    mu, sigma = 6e-4, 0.018   # Günlük drift ve vol

    # Volatilite kümelenmesi: GARCH(1,1) benzeri
    eps = rng.standard_normal(n_days)
    vol = np.zeros(n_days)
    vol[0] = sigma
    for t in range(1, n_days):
        vol[t] = np.sqrt(max(1e-6,
            sigma**2 * 0.05 + 0.10 * (vol[t-1]*eps[t-1])**2 + 0.85 * vol[t-1]**2))
    log_r = mu + vol * eps

    close  = 50 * np.exp(np.cumsum(log_r))
    intra  = 0.012
    high   = close * np.exp( abs(rng.normal(0, intra, n_days)))
    low    = close * np.exp(-abs(rng.normal(0, intra, n_days)))
    open_  = close * np.exp(rng.normal(0, intra*0.3, n_days))
    volume = (800_000 * (1 + abs(log_r)/sigma) * rng.lognormal(0, 0.4, n_days)).astype(int)

    idx_lr  = 0.65*log_r + 0.35*(4e-4 + 0.014*rng.standard_normal(n_days))
    endeks  = 8500 * np.exp(np.cumsum(idx_lr))

    dates = pd.bdate_range("2019-01-02", periods=n_days, freq="B")
    df    = pd.DataFrame({
        "Open": open_, "High": high, "Low": low,
        "Close": close, "Volume": volume, "Endeks_Close": endeks,
    }, index=dates)
    df.index.name = "Date"

    print(f"Sentetik veri: {n_days} gün | "
          f"{dates[0].date()} → {dates[-1].date()}")
    print(f"  Close: {close[0]:.2f} → {close[-1]:.2f}  "
          f"(Kümülatif: {(close[-1]/close[0]-1)*100:+.1f}%)")
    return df


# ── Demo için veri oluştur ───────────────────────────────────────────────────
df = create_synthetic_bist(n_days=1500, seed=42)
df.tail(3)

In [ ]:
# ── Gerçek BIST Verisi (opsiyonel) ──────────────────────────────────────────
# RUN_YFINANCE = True yaparak gerçek hisse verisi çekebilirsiniz.
# Bu hücre çalıştırılırsa demo df üzerine yazar.

RUN_YFINANCE = False

if RUN_YFINANCE:
    import yfinance as yf

    HISSE  = "THYAO.IS"   # ← İstediğiniz hisse
    ENDEKS = "^XU100"
    PERIOD = "7y"

    h  = yf.download(HISSE,  period=PERIOD, auto_adjust=True, progress=False)
    xe = yf.download(ENDEKS, period=PERIOD, auto_adjust=True, progress=False)

    for frame in [h, xe]:
        if isinstance(frame.columns, pd.MultiIndex):
            frame.columns = frame.columns.get_level_values(0)

    df = h[["Open","High","Low","Close","Volume"]].copy()
    df["Endeks_Close"] = xe["Close"]
    df.dropna(inplace=True)
    df.index.name = "Date"

    print(f"Gerçek veri yüklendi: {HISSE}")
    print(f"  {len(df)} gün | {df.index[0].date()} → {df.index[-1].date()}")
    print(f"  Son fiyat: {df['Close'].iloc[-1]:.2f} TL")

## Çalıştır — Tam Analiz

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# TÜMÜNÜ ÇALIŞTIR
# ════════════════════════════════════════════════════════════════════════════
# Beklenen süre: ~5-15 dk (veri büyüklüğüne bağlı; GARCH fold başına ~5-8 sn)

# 1. Walk-Forward Analiz
df_results, champion, wfo_positions, df_feat = run_wfo_comparison(df)

# 2. Şampiyon Stratejinin Güncel Sinyali
signal_info = generate_current_signal(df_feat, champion, STRATEGIES)